# File này xử lý việc Phân tích và Tiền xử lý Dữ liệu, các file khác dùng file dataset.csv để dùng data đã được xử lý


## 1. Mô tả dữ liệu

In [ ]:
#import library
import pandas as pd # pandas
import numpy as np # numpy
import time

In [ ]:
# read data using Pandas DataFrame
def read_dataset(path):
    # read data using Pandas DataFrame
    df = pd.read_excel(path)
    display(df.head())
    print("\nDataFrame Info:")
    print(df.info())
    print("\nDataFrame Shape:", df.shape)
    display(df.describe())
    return df

In [ ]:
# 1. MÔ TẢ DỮ LIỆU
# Đọc file Excel
df = read_dataset('data/Dry_Bean_Dataset.xlsx')


## Data Analysis

### 1. Checking for Missing Data
It is essential to check for null values before modeling. We use `.isnull().sum()` to identify any columns that require data imputation.

In [ ]:
# Check for missing values in the dataset
print("Missing values summary:")
display(df.isnull().sum())

## 2. Phân tích khám phá (EDA):

### 1. Thống kê mô tả: mean, std, min, max, tứ phân vị.

In [ ]:
# Thống kê mô tả các biến số
print(df.describe())


### 2. Phân bố của biến mục tiêu (histogram, boxplot).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập phong cách cho đồ thị
sns.set_theme(style="whitegrid")

# Khởi tạo khung hình gồm 1 hàng và 2 cột
fig, axes = plt.subplots(3, 1, figsize=(14, 20))

# 1. Vẽ Histogram (Biểu đồ tần suất) kèm đường mật độ KDE
sns.histplot(df['Area'], kde=True, ax=axes[0], color='skyblue', bins=100)
axes[0].set_title('Histogram & KDE của biến Area', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Diện tích (Area)')
axes[0].set_ylabel('Tần suất')

# 2. Vẽ Boxplot (Biểu đồ hộp) để phát hiện ngoại lai
sns.boxplot(x=df['Class'], y=df['Area'], ax=axes[1], color='lightcoral')
axes[1].set_title('Boxplot của biến Area', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Loại (Class)')
axes[1].set_ylabel('Diện tích (Area)')

# 3. Vẽ biểu đồ phân tán (Scatter Plot) giữa 'Area' và 'Perimeter' để xem mối quan hệ giữa hai biến
sns.scatterplot(x=df['Area'], y=df['Perimeter'], hue=df['Class'], ax=axes[2])
axes[2].set_title('Scatter Plot của biến Area và Perimeter', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Diện tích (Area)')
axes[2].set_ylabel('Chu vi (Perimeter)')

# Căn chỉnh bố cục
plt.tight_layout()
plt.show()

In [ ]:
# Kiểm tra số lượng mẫu của từng loại đậu
print("\nSố lượng mẫu mỗi loại đậu:")
print(df['Class'].value_counts())

# Vẽ biểu đồ cột để trực quan hóa độ cân bằng
plt.figure(figsize=(10, 5))
sns.countplot(x='Class', data=df, palette='viridis', order=df['Class'].value_counts().index)
plt.title('Phân bố các loại đậu trong tập dữ liệu', fontsize=14, fontweight='bold')
plt.xlabel('Loại đậu')
plt.ylabel('Số lượng mẫu')
plt.xticks(rotation=45)
plt.show()

# Tính tỷ lệ phần trăm
print("\nTỷ lệ phần trăm các lớp:")
print(df['Class'].value_counts(normalize=True) * 100)

### 3. Ma trận tương quan

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title("Ma trận tương quan")
plt.show()

"Qua ma trận tương quan, nhóm nghiên cứu nhận thấy hiện tượng đa cộng tuyến nghiêm trọng giữa các biến kích thước (Area, Perimeter, Diameter với $r > 0.95$). Để tối ưu hóa mô hình Logistic Regression và đảm bảo tính ổn định của ma trận Hessian khi sử dụng solver Newton-Raphson, chúng tôi quyết định loại bỏ các biến dư thừa và chỉ giữ lại các đại diện tiêu biểu như Area, AspectRation và Solidity."

### 4. Phát hiện ngoại lai (outliers) bằng phương pháp IQR hoặc Z-score. 

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 20))

sns.histplot(df['Area'], kde=True, ax=axes[0], color='skyblue', bins=100)
axes[0].set_title('Histogram & KDE của biến Area')

sns.boxplot(x='Class', y='Area', data=df, ax=axes[1], color='lightcoral')
axes[1].set_title('Boxplot của biến Area')
axes[1].set_xlabel('Loại (Class)')
axes[1].set_ylabel('Diện tích (Area)')

plt.tight_layout()
plt.show()

In [ ]:
# Tách dữ liệu thành đặc trưng (X) và mục tiêu (y) TRƯỚC khi lọc đa cộng tuyến
X = df.drop(columns=['Class'])
y = df['Class']

def reduce_multicollinearity(df, threshold=0.95):
    corr_matrix = df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    
    print(f"Các cột bị loại bỏ do đa cộng tuyến (> {threshold}): {to_drop}")
    return df.drop(columns=to_drop)

# Áp dụng lọc trên X
X_reduced = reduce_multicollinearity(X, threshold=0.95)
X = X_reduced # Cập nhật lại X

In [ ]:
# Xử lý Outliers bằng phương pháp Capping (giữ nguyên số lượng dòng)
for col in X.select_dtypes(include=[np.number]).columns:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Thay thế các giá trị vượt ngưỡng bằng giá trị biên
    X[col] = np.where(X[col] < lower_bound, lower_bound, X[col])
    X[col] = np.where(X[col] > upper_bound, upper_bound, X[col])

print("Đã xử lý ngoại lai cho các đặc trưng số.")
print(X)

## 3. Tiền xử lý:

### 1. Xử lý giá trị còn thiếu (missing values): nếu có


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 1. Mã hóa biến mục tiêu y
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 2. Tạo DataFrame đã làm sạch để xuất file CSV (không bao gồm các cột đa cộng tuyến)
df_cleaned = X.copy()
df_cleaned['Class'] = y.values
df_cleaned['Class_Encoded'] = y_encoded
df_cleaned.to_csv('data/dataset.csv', index=False)
print("Đã lưu dữ liệu sạch vào 'data/dataset.csv'")
print(y_encoded)

### 2. Chia tập dữ liệu

In [ ]:
# 3. Chia tập dữ liệu (Lưu ý: Stratify theo y để cân bằng tỷ lệ các loại đậu)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125, random_state=42, stratify=y_train_val
)

print(f"Kích thước Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

### 3. Chuẩn hóa dữ liệu

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# Chỉ fit trên tập Train để tránh Leakage
X_train_scaled = scaler.fit_transform(X_train)

# Transform tập Val và Test dựa trên thông số của tập Train
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Đã chuẩn hóa dữ liệu thành công.")